# 06. yolo26m_1280_seed42 - person A

Attach the UPDATED manual code Dataset, enable Internet and GPU, then **Run All**. Prepared VisDrone is optional: Setup reuses it or prepares the original official splits automatically.
Model/seed/size are fixed. Training, evaluation and final export run in order. No task queue, API keys or cloud saving.

## A. Setup
Code is verified and copied into one writable project folder. Setup checks GPU, resolves local/prepared data (folder or ZIP), or prepares it on CPU, and checks official COCO weights. GPU T4 x2 is supported; this baseline protocol uses GPU0.

In [ ]:
import hashlib
import json
import os
import sys
import zipfile
from pathlib import Path, PurePosixPath

CODE = Path("/kaggle/working/small-object-detection-manual")
inputs = Path("/kaggle/input")
manifests = sorted(inputs.rglob("manual_code_manifest.json"))
if manifests:
    if len(manifests) != 1:
        raise RuntimeError("Attach one code Dataset; several code manifests were found.")
    source = manifests[0].parent
    manifest = json.loads(manifests[0].read_text(encoding="utf-8"))
    archive_path = None
else:
    bundles = []
    for path in sorted(inputs.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            for name in z.namelist():
                if PurePosixPath(name).name == "manual_code_manifest.json":
                    bundles.append((path, name))
    if len(bundles) != 1:
        raise RuntimeError("Add Input: attach the updated manual_kaggle_code.zip code Dataset.")
    archive_path, manifest_name = bundles[0]
    prefix = manifest_name[: -len("manual_code_manifest.json")]
    with zipfile.ZipFile(archive_path) as z:
        if z.testzip():
            raise ValueError("Code ZIP failed its integrity check.")
        manifest = json.loads(z.read(manifest_name))
if manifest.get("notebook_mode") != "run_all_v1":
    raise RuntimeError(
        "This is the old code Dataset. Upload the updated manual_kaggle_code.zip and select its new version."
    )
files = manifest["files"]
for required in (
    "scripts/manual_experiment.py",
    "scripts/prepare_data.py",
    "notebooks/baseline.ipynb",
    "src/evaluation/metrics.py",
):
    if required not in files:
        raise ValueError(f"Incomplete code manifest: {required}")
verified = {}
for name, digest in files.items():
    relative = PurePosixPath(name)
    if relative.is_absolute() or ".." in relative.parts or "\\" in name or ":" in name:
        raise ValueError(f"Unsafe manifest path: {name}")
    if archive_path is None:
        data = (source / name).read_bytes()
    else:
        with zipfile.ZipFile(archive_path) as z:
            data = z.read(prefix + name)
    if hashlib.sha256(data).hexdigest() != digest:
        raise ValueError(f"Missing or corrupt code file: {name}")
    verified[name] = data
# Only manifest-listed code files are copied. Existing data and run results are preserved.
for name, data in verified.items():
    target = CODE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != files[name]:
        target.write_bytes(data)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
os.chdir("/kaggle/working")
print(f"CODE READY: {CODE}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.175", "pycocotools==2.0.11"],
    check=True,
)

In [ ]:
import subprocess
import sys
from pathlib import Path

code = Path("/kaggle/working/small-object-detection-manual")
subprocess.run(
    [sys.executable, str(code / "scripts/manual_experiment.py"), "setup", "--model", "yolo26m"],
    check=True,
)

### Optional shared prepared-data ZIP
Run All skips this. To save a reusable Dataset after Setup, set `CREATE_PREPARED_ZIP=True`, execute this cell, download the ZIP and upload it once as a shared Kaggle Dataset.

In [ ]:
CREATE_PREPARED_ZIP = False  # Set True only for this optional/manual action.
if CREATE_PREPARED_ZIP:
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    import os

    from IPython.display import FileLink, display

    result = subprocess.check_output(
        [sys.executable, str(code / "scripts/manual_experiment.py"), "prepare"], text=True
    )
    print(result)
    display(FileLink(os.path.relpath(result.strip().splitlines()[-1])))
else:
    print("Skipped: optional/manual action, or Training is not complete.")

## D. Manual Resume (skipped by default)
For a new session with a downloaded unfinished-run Backup attached: set `RESTORE_BACKUP=True`, then Run All. This restores before Training. It checks run identity, optimizer and scheduler; it never silently starts fresh when restore fails. Keep False for fresh experiments.

In [ ]:
RESTORE_BACKUP = False  # Set True only for this optional/manual action.
if RESTORE_BACKUP:
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    run = Path("/kaggle/working/manual_experiments/yolo26m_1280_seed42")
    if run.exists():
        print(
            "Local run already exists; it will be validated by Training, not overwritten by Restore."
        )
    else:
        subprocess.run(
            [
                sys.executable,
                str(code / "scripts/manual_experiment.py"),
                "resume",
                "--experiment",
                "yolo26m_1280_seed42",
            ],
            check=True,
        )
else:
    print("Skipped: optional/manual action, or Training is not complete.")

### Memory pilot
Automatically probes the fixed 1280 experiment before fresh Training. Reuses a validated batch decision after restore/repeated Run All. Full Training always starts independently from COCO weights.

In [ ]:
from pathlib import Path

if Path("/kaggle/working/manual_experiments/yolo26m_1280_seed42_pilot.json").is_file():
    print("Pilot decision already saved; Training validates it.")
else:
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    subprocess.run(
        [
            sys.executable,
            str(code / "scripts/manual_experiment.py"),
            "pilot",
            "--experiment",
            "yolo26m_1280_seed42",
        ],
        check=True,
    )

## B. Training
Max50 epochs, patience15; original baseline args and fixed physical batch. Completed runs are checked and skipped on repeated Run All.

To pause: Interrupt this cell once, wait for `STOPPED SAFELY` and completion of the epoch/checkpoint. Later evaluation/export cells will skip the unfinished run. Do not terminate the session before manually downloading Backup.

In [ ]:
import sys
from pathlib import Path

code = Path("/kaggle/working/small-object-detection-manual")
if str(code) not in sys.path:
    sys.path.insert(0, str(code))
from scripts.manual_experiment import run_notebook

run_notebook(
    [
        sys.executable,
        str(code / "scripts/manual_experiment.py"),
        "train",
        "--experiment",
        "yolo26m_1280_seed42",
    ],
    "/kaggle/working/manual_experiments/yolo26m_1280_seed42_training.log",
)

## E. Evaluation
Run All executes full640/full1280 and, for training640, tiled640 overlap20%. Completed evaluation outputs are SHA256 checked and reused. Original evaluator/thresholds/ignore handling/cap are unchanged; new CSV/runtime/provenance and figures stay inside this run.

### Full640

In [ ]:
from pathlib import Path

if Path("/kaggle/working/manual_experiments/yolo26m_1280_seed42/completed.json").is_file():
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    subprocess.run(
        [
            sys.executable,
            str(code / "scripts/manual_experiment.py"),
            "evaluate",
            "--experiment",
            "yolo26m_1280_seed42",
            "--mode",
            "full-640",
        ],
        check=True,
    )
else:
    print("Skipped: optional/manual action, or Training is not complete.")

### Full1280

In [ ]:
from pathlib import Path

if Path("/kaggle/working/manual_experiments/yolo26m_1280_seed42/completed.json").is_file():
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    subprocess.run(
        [
            sys.executable,
            str(code / "scripts/manual_experiment.py"),
            "evaluate",
            "--experiment",
            "yolo26m_1280_seed42",
            "--mode",
            "full-1280",
        ],
        check=True,
    )
else:
    print("Skipped: optional/manual action, or Training is not complete.")

### Tiled640 overlap20% (optional for training1280)

In [ ]:
RUN_OPTIONAL_TILING = False  # Set True only for this optional/manual action.
from pathlib import Path

if (
    RUN_OPTIONAL_TILING
    and Path("/kaggle/working/manual_experiments/yolo26m_1280_seed42/completed.json").is_file()
):
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    subprocess.run(
        [
            sys.executable,
            str(code / "scripts/manual_experiment.py"),
            "evaluate",
            "--experiment",
            "yolo26m_1280_seed42",
            "--mode",
            "tiled-ov20",
        ],
        check=True,
    )
else:
    print("Skipped: optional/manual action, or Training is not complete.")

## F. Final Export
Run All creates a checked download ZIP after completed Training/evaluation. It contains checkpoints, configs, training logs, evaluation CSV/provenance and figures. Download it before ending the session.

In [ ]:
from pathlib import Path

if Path("/kaggle/working/manual_experiments/yolo26m_1280_seed42/completed.json").is_file():
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    import os

    from IPython.display import FileLink, display

    result = subprocess.check_output(
        [
            sys.executable,
            str(code / "scripts/manual_experiment.py"),
            "export",
            "--experiment",
            "yolo26m_1280_seed42",
        ],
        text=True,
    )
    print(result)
    display(FileLink(os.path.relpath(result.strip().splitlines()[-1])))
else:
    print("Skipped: optional/manual action, or Training is not complete.")

## C. Manual Backup (skipped by default)
After a safe pause, set `CREATE_BACKUP=True` and execute only this cell. It does not rerun Training. Download the ZIP before ending the session. Missing/corrupt optimizer-bearing last.pt, inconsistent scheduler or an active writer are rejected. Finished/stripped weights use Final Export instead.

In [ ]:
CREATE_BACKUP = False  # Set True only for this optional/manual action.
if CREATE_BACKUP:
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    import os

    from IPython.display import FileLink, display

    result = subprocess.check_output(
        [
            sys.executable,
            str(code / "scripts/manual_experiment.py"),
            "backup",
            "--experiment",
            "yolo26m_1280_seed42",
        ],
        text=True,
    )
    print(result)
    display(FileLink(os.path.relpath(result.strip().splitlines()[-1])))
else:
    print("Skipped: optional/manual action, or Training is not complete.")

Protocol limits: class-agnostic localization recall; simplified ignore filtering using box IoU; COCO-style AP with custom cap1000. A memory pilot cannot guarantee that every crowded training image fits. New sessions require a downloaded Backup to resume unfinished training.